In [1]:
import torch
import torch.nn as nn

model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
print(model)


Using cache found in C:\Users\Ardbert Conner/.cache\torch\hub\facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "c:\ProgramData\miniconda3\envs\rgb\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'
C:\Users\Ardbert Conner/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
C:\Users\Ardbert Conner/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
C:\Users\Ardbert Conner/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [9]:
old_conv = model.patch_embed.proj

new_conv = nn.Conv2d(
    in_channels=1, 
    out_channels=old_conv.out_channels, 
    kernel_size=old_conv.kernel_size, 
    stride=old_conv.stride, 
    padding=old_conv.padding
)
with torch.no_grad():
    new_weight = old_conv.weight.sum(dim=1, keepdim=True) / 3.0
    new_conv.weight.copy_(new_weight)
    new_conv.bias.copy_(old_conv.bias)

model.patch_embed.proj = new_conv


In [2]:
model.to('cuda')

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [10]:
input_gray = torch.randn(1, 3, 518, 518).to('cuda')
features = model.get_intermediate_layers(input_gray, [2, 5, 8, 11], return_class_token=True, reshape=True) 


In [11]:
features[0][0].shape

torch.Size([1, 384, 37, 37])

In [15]:
import torch
from sdt_head import SDTHead

# in_channels: ViT-S=384, ViT-B=768, ViT-L=1024
# extract layers: [2,5,8,11] for ViT-S/B, [4,11,17,23] for ViT-L
head = SDTHead(
    in_channels=[384, 384, 384, 384],
    fusion_channels=256,
    n_output_channels=3,
    use_cls_token=True
)
head.to('cuda')

SDTHead(
  (weighted_fusion): WeightedFusion(
    (projections): ModuleList(
      (0-3): 4 x Sequential(
        (0): Linear(in_features=384, out_features=256, bias=False)
        (1): GELU(approximate='none')
      )
    )
    (readout_projects): ModuleList(
      (0-3): 4 x Sequential(
        (0): Linear(in_features=768, out_features=384, bias=True)
        (1): GELU(approximate='none')
      )
    )
  )
  (detail_enhancer): SpatialDetailEnhancer(
    (dwconv): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=False)
    (norm): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (activation): ReLU(inplace=True)
  )
  (upsample_1): DySampleUpsamplerWrapper(
    (dysample1): Sequential(
      (0): DySample(
        (offset): Conv2d(256, 32, kernel_size=(1, 1), stride=(1, 1))
        (scope): Conv2d(256, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      )
      (1): Conv2d(256, 256, kernel_size=(3, 3), stride=

In [17]:
y = head(features)

In [18]:
y.shape

torch.Size([1, 3, 592, 592])